# Week 32, Day 2: Enums

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w032d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from enum import Enum

class Light(Enum):
    RED = 1
    AMBER = 2
    GREEN = 3

print(Light.RED, Light.RED.name, Light.RED.value)
print(Light(3), Light["AMBER"], len(Light), Light.RED == 1)

In [ ]:
def can_go(light):
    return light == "green"

print(can_go("green"), can_go("Green"), can_go("grren"), can_go(3))

In [ ]:
from enum import Enum

class Light(Enum):
    RED = "red"
    AMBER = "amber"
    GREEN = "green"

def can_go(light):
    return light is Light.GREEN

print(can_go(Light.GREEN), can_go(Light.RED))
try:
    can_go(Light.GREEN_)
except AttributeError as err:
    print("AttributeError:", err)
print(list(Light), Light.AMBER in Light)

In [ ]:
from enum import Enum

class Size(Enum):
    SMALL = "S"
    MEDIUM = "M"
    LARGE = "L"

print(Size("M"), Size["LARGE"])       # by value, by name
for text in ["L", "XL", "large"]:
    try:
        print(text, "->", Size(text))
    except ValueError as err:
        print(text, "-> ValueError:", err)
try:
    Size["Large"]
except KeyError as err:
    print("KeyError:", err)

In [ ]:
from enum import Enum

class Day(Enum):
    MON = 1
    TUE = 2
    WED = 3
    THU = 4
    FRI = 5
    SAT = 6
    SUN = 7

    @property
    def is_weekend(self):
        return self in (Day.SAT, Day.SUN)

    def next(self):
        days = list(Day)
        return days[(days.index(self) + 1) % len(days)]

    def __str__(self):
        return self.name.title()

print(Day.SAT.is_weekend, Day.TUE.is_weekend, Day.SUN.next(), Day.WED.next().next())
print([str(d) for d in Day if not d.is_weekend])

In [ ]:
from enum import Enum

class Size(Enum):
    SMALL = "S"
    MEDIUM = "M"
    LARGE = "L"

PRICES = {Size.SMALL: 250, Size.MEDIUM: 320, Size.LARGE: 400}

order = [Size.LARGE, Size.SMALL, Size.LARGE]
print(sum(PRICES[s] for s in order))

In [ ]:
from enum import Enum, auto

class Stage(Enum):
    PLACED = auto()
    PAID = auto()
    SHIPPED = auto()

print([(s.name, s.value) for s in Stage])

In [ ]:
import json
from dataclasses import dataclass, field
from enum import Enum

class Stage(Enum):
    PLACED = "placed"
    PAID = "paid"
    SHIPPED = "shipped"

@dataclass
class Order:
    code: str
    stage: Stage = Stage.PLACED
    history: list = field(default_factory=list)

    def advance(self):
        stages = list(Stage)
        i = stages.index(self.stage)
        if i == len(stages) - 1:
            raise ValueError(f"{self.code} is already {self.stage.value}")
        self.history.append(self.stage)
        self.stage = stages[i + 1]

    def to_json(self):
        return json.dumps({"code": self.code, "stage": self.stage.value})

o = Order("A17")
o.advance()
print(o.stage, o.history, o.to_json())
loaded = json.loads(o.to_json())
print(Order(loaded["code"], Stage(loaded["stage"])))

In [ ]:
from enum import IntEnum, StrEnum, Flag, auto

class Priority(IntEnum):
    LOW = 1
    NORMAL = 2
    URGENT = 3

class Colour(StrEnum):
    RED = "red"
    BLUE = "blue"

class Perm(Flag):
    READ = auto()
    WRITE = auto()
    DELETE = auto()

print(Priority.URGENT > Priority.LOW, Priority.NORMAL == 2, sorted([Priority.URGENT, Priority.LOW]))
print(Colour.RED == "red", "colour: " + Colour.BLUE)
editor = Perm.READ | Perm.WRITE
print(editor, Perm.WRITE in editor, Perm.DELETE in editor)

> **Key idea.** An Enum is a type with a fixed set of named members: `class Light(Enum): RED = 1`. Members have a `name` and a `value`, compare with `is`, iterate in order, are hashable, and are never equal to their plain values. Look them up with `Light(value)` (`ValueError`) or `Light[name]` (`KeyError`) at the edges of the program, and save them as `.value` or `.name`. Enums can have methods and properties; plain members aren't ordered. `IntEnum` and `StrEnum` members are also numbers or strings, and `Flag` members combine with `|`.

### Exercise: A UK traffic light

In the UK a traffic light goes red, red and amber together, green, amber, then red again. Write an Enum `Light` with members `RED`, `RED_AMBER`, `GREEN` and `AMBER`, in that order, with the values `"red"`, `"red and amber"`, `"green"` and `"amber"`. Give it a method `next()` that returns the following member (after `AMBER` comes `RED`), and a property `can_go`, True only for `GREEN`.

In [ ]:
from enum import Enum

class Light(Enum):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_light():\n    \"the sequence\"\n    L = ns[\"Light\"]\n    assert [m.name for m in L] == [\"RED\", \"RED_AMBER\", \"GREEN\", \"AMBER\"]\n    assert L(\"red and amber\") is L.RED_AMBER and L.RED.next() is L.RED_AMBER and L.AMBER.next() is L.RED\n    seq, m = [], L.GREEN\n    for _ in range(5):\n        seq.append(m.value)\n        m = m.next()\n    assert seq == [\"green\", \"amber\", \"red\", \"red and amber\", \"green\"]\n    assert L.GREEN.can_go is True and [l.can_go for l in L].count(True) == 1")

### Exercise: Reading days of the week

`Day` is given. Write `parse_day(text)`, which accepts a day's three-letter name (`"tue"`) or full English name (`"Tuesday"`), in any case and with spaces at the ends, and returns the member. Anything else raises `ValueError` (not `KeyError`). Then write `weekdays_between(a, b)`: the number of days from `a` up to, but not including, `b` that aren't Saturday or Sunday, going forward through the week and wrapping round (from `FRI` to `TUE` is `FRI`, `SAT`, `SUN`, `MON`: 2 weekdays). From a day to itself is 0.

In [ ]:
from enum import Enum

class Day(Enum):
    MON = 1
    TUE = 2
    WED = 3
    THU = 4
    FRI = 5
    SAT = 6
    SUN = 7

FULL = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

def parse_day(text):
    ___

def weekdays_between(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parse():\n    \"parsing\"\n    D = ns[\"Day\"]\n    assert parse_day(\"tue\") is D.TUE and parse_day(\"  Tuesday \") is D.TUE and parse_day(\"SUN\") is D.SUN and parse_day(\"friday\") is D.FRI\n    for bad in [\"tues\", \"t\", \"\", \"funday\", \"7\", \"TUESDAY!\"]:\n        try:\n            parse_day(bad)\n            assert False, f\"{bad!r} should raise ValueError\"\n        except ValueError:\n            pass\ndef test_between():\n    \"counting weekdays\"\n    D = ns[\"Day\"]\n    assert weekdays_between(D.MON, D.FRI) == 4 and weekdays_between(D.FRI, D.TUE) == 2\n    assert weekdays_between(D.SAT, D.MON) == 0 and weekdays_between(D.WED, D.WED) == 0 and weekdays_between(D.SUN, D.SAT) == 5")

### Exercise: Change in coins

Write an Enum `Coin` for the eight UK coins, with values in pence: `P1 = 1`, `P2 = 2`, `P5 = 5`, `P10 = 10`, `P20 = 20`, `P50 = 50`, `POUND = 100`, `TWO_POUNDS = 200`. Give it a property `label`: `"1p"` up to `"50p"`, then `"£1"` and `"£2"`. Then write `change(pence)`: a dictionary from `Coin` to how many of it, using as many of the biggest coins as possible first, leaving out coins not used. Negative amounts raise `ValueError`.

In [ ]:
from enum import Enum

class Coin(Enum):
    ___

def change(pence):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_coins():\n    \"coins\"\n    C = ns[\"Coin\"]\n    assert len(C) == 8 and C(50) is C.P50 and [c.label for c in C] == [\"1p\", \"2p\", \"5p\", \"10p\", \"20p\", \"50p\", \"\u00a31\", \"\u00a32\"]\n    assert change(388) == {C.TWO_POUNDS: 1, C.POUND: 1, C.P50: 1, C.P20: 1, C.P10: 1, C.P5: 1, C.P2: 1, C.P1: 1}\n    assert change(400) == {C.TWO_POUNDS: 2} and change(0) == {} and change(9) == {C.P5: 1, C.P2: 2}\n    try:\n        change(-1)\n        assert False, \"negative amounts should raise ValueError\"\n    except ValueError:\n        pass")

### Exercise: An order's allowed moves

An order moves through stages, but not every move is allowed. Write an Enum `Stage` (`PLACED`, `PAID`, `SHIPPED`, `DELIVERED`, `CANCELLED`, with lower-case strings as values) and a dataclass `Order` with `code`, `stage` (default `Stage.PLACED`) and `history` (a new list for each order). `move(to)` changes the stage if the move is allowed, recording the old stage in `history`, and raises `ValueError` otherwise. The allowed moves are:

- `PLACED` to `PAID` or `CANCELLED`;
- `PAID` to `SHIPPED` or `CANCELLED`;
- `SHIPPED` to `DELIVERED`;
- nothing from `DELIVERED` or `CANCELLED`.

`move` also accepts the stage as its value (a string like `"paid"`), turning it into a member (an unknown string raises `ValueError`).

In [ ]:
from dataclasses import dataclass, field
from enum import Enum

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_moves():\n    \"moves\"\n    S, O = ns[\"Stage\"], ns[\"Order\"]\n    o = O(\"A17\")\n    assert o.stage is S.PLACED\n    o.move(S.PAID)\n    o.move(\"shipped\")\n    o.move(S.DELIVERED)\n    assert o.stage is S.DELIVERED and o.history == [S.PLACED, S.PAID, S.SHIPPED] and O(\"B\").history == []\n    for start, bad in [(S.DELIVERED, S.CANCELLED), (S.PLACED, S.SHIPPED), (S.SHIPPED, S.CANCELLED)]:\n        x = O(\"X\", start)\n        try:\n            x.move(bad)\n            assert False, f\"{start.name} to {bad.name} isn't allowed\"\n        except ValueError:\n            pass\n        assert x.stage is start and x.history == []\n    c = O(\"C\")\n    c.move(\"cancelled\")\n    assert c.stage is S.CANCELLED\n    try:\n        O(\"D\").move(\"lost\")\n        assert False, \"an unknown stage should raise ValueError\"\n    except ValueError:\n        pass")

### Bug hunt: Never stops

The barrier should open only for the green light, so this should print `closed`, `closed`, `open`, but it prints `closed` three times. Fix the comparison.

In [ ]:
from enum import Enum

class Light(Enum):
    RED = "red"
    AMBER = "amber"
    GREEN = "green"

def barrier(light):
    return "open" if light == "green" else "closed"

for light in Light:
    print(barrier(light))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"only green opens\"\n    expect_output(\"closed\\nclosed\\nopen\")")

### Bug hunt: Name or value?

The shop's orders file stores sizes by **name**, and loading it crashes with `ValueError: 'LARGE' is not a valid Size`. Fix the lookup so it prints `[<Size.LARGE: 'L'>, <Size.SMALL: 'S'>]`.

In [ ]:
from enum import Enum

class Size(Enum):
    SMALL = "S"
    MEDIUM = "M"
    LARGE = "L"

saved = ["LARGE", "SMALL"]
print([Size(name) for name in saved])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"loaded\"\n    expect_output(\"[<Size.LARGE: 'L'>, <Size.SMALL: 'S'>]\")")

### Bug hunt: The missing shift

A rota has three shifts, but this prints `2 shifts: EARLY, LATE`. Fix the Enum so all three are real members and it prints `3 shifts: EARLY, LATE, NIGHT`, and make Python refuse duplicate values in future.

In [ ]:
from enum import Enum, unique

class Shift(Enum):
    EARLY = 1
    LATE = 2
    NIGHT = 2

print(f"{len(Shift)} shifts: {', '.join(s.name for s in Shift)}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"three shifts\"\n    expect_output(\"3 shifts: EARLY, LATE, NIGHT\")\n    assert \"@unique\" in source, \"use @unique\"")

### Puzzle: Suits with colours

Arrange the lines into an Enum of card suits with a `colour` property. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from enum import Enum, IntEnum

class A(Enum):
    X = 1
    Y = 2
    Z = 1

class B(IntEnum):
    LOW = 1
    HIGH = 5

p = ___   # len(A)
q = ___   # A.Z is A.X
r = ___   # A.X == 1
s = ___   # B.HIGH > B.LOW
t = ___   # B.HIGH + 1
u = ___   # A(2).name

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (2, True, False), \"p to r\"\n    assert (s, t, u) == (True, 6, \"Y\")")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: naming poker hands

Classify five-card poker hands using Enums and a dataclass.

- `Suit(Enum)`: `CLUBS = "♣"`, `DIAMONDS = "♦"`, `HEARTS = "♥"`, `SPADES = "♠"`.
- `Rank(IntEnum)`: `TWO = 2` up to `TEN = 10`, then `JACK = 11`, `QUEEN = 12`, `KING = 13`, `ACE = 14`. A property `symbol` gives `"2"` to `"10"`, `"J"`, `"Q"`, `"K"`, `"A"`.
- `Card`, a frozen dataclass with `rank` and `suit`. `str` gives `Q♥`. The class method `parse(text)` reads `"Q♥"`, `"10♠"` or `"qh"` (the suit may also be the letter `C`, `D`, `H` or `S`, and letters in any case); anything else raises `ValueError`.
- `Hand(IntEnum)`, from lowest to highest: `HIGH_CARD`, `PAIR`, `TWO_PAIR`, `THREE`, `STRAIGHT`, `FLUSH`, `FULL_HOUSE`, `FOUR`, `STRAIGHT_FLUSH` (values 1 to 9).
- `classify(cards)` takes five cards and returns the `Hand`. A **straight** is five consecutive ranks (the ace may also count low, in `A 2 3 4 5`, but not in the middle, as in `Q K A 2 3`); a **flush** is five of one suit; a **full house** is three of one rank and two of another. Fewer or more than five cards, or the same card twice, raises `ValueError`.

Because `Hand` is an `IntEnum`, `classify(a) > classify(b)` says which kind of hand is better.

In [ ]:
from dataclasses import dataclass
from enum import Enum, IntEnum

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def cards(text):\n    return [ns[\"Card\"].parse(t) for t in text.split()]\ndef test_cards():\n    \"cards\"\n    C, R, S = ns[\"Card\"], ns[\"Rank\"], ns[\"Suit\"]\n    q = C.parse(\"Q\u2665\")\n    assert q == C(R.QUEEN, S.HEARTS) and str(q) == \"Q\u2665\" and str(C.parse(\"10s\")) == \"10\u2660\" and str(C.parse(\" ac \")) == \"A\u2663\"\n    assert [r.symbol for r in R][:3] == [\"2\", \"3\", \"4\"] and R.TEN.symbol == \"10\" and R.KING.symbol == \"K\" and len(R) == 13\n    for bad in [\"1\u2665\", \"11h\", \"Q\", \"QX\", \"\", \"\u2665Q\"]:\n        try:\n            C.parse(bad)\n            assert False, f\"{bad!r} should raise ValueError\"\n        except ValueError:\n            pass\ndef test_classify():\n    \"classifying\"\n    H = ns[\"Hand\"]\n    cases = {\n        \"2h 7d 9c Js Kh\": H.HIGH_CARD, \"2h 2d 9c Js Kh\": H.PAIR, \"2h 2d 9c 9s Kh\": H.TWO_PAIR,\n        \"9h 2d 9c 9s Kh\": H.THREE, \"5h 6d 7c 8s 9h\": H.STRAIGHT, \"Ah 2d 3c 4s 5h\": H.STRAIGHT,\n        \"10h Jd Qc Ks Ah\": H.STRAIGHT, \"Qh Kd Ac 2s 3h\": H.HIGH_CARD, \"2h 7h 9h Jh Kh\": H.FLUSH,\n        \"9h 9d 9c Ks Kh\": H.FULL_HOUSE, \"9h 9d 9c 9s Kh\": H.FOUR, \"5s 6s 7s 8s 9s\": H.STRAIGHT_FLUSH,\n        \"As 2s 3s 4s 5s\": H.STRAIGHT_FLUSH,\n    }\n    for text, want in cases.items():\n        got = classify(cards(text))\n        assert got is want, f\"{text}: got {got.name}, expected {want.name}\"\n    assert classify(cards(\"9h 9d 9c Ks Kh\")) > classify(cards(\"2h 7h 9h Jh Kh\")) and len(H) == 9 and H.PAIR == 2\n    for text in [\"2h 3h 4h 5h\", \"2h 3h 4h 5h 6h 7h\", \"2h 2h 4d 5c 9s\"]:\n        try:\n            classify(cards(text))\n            assert False, f\"{text} should raise ValueError\"\n        except ValueError:\n            pass")

## Warm-up

`Light.RED` isn't the number 1; it's a **member** of the `Light` type, with a **name** (`"RED"`) and a **value** (`1`). You can get members back from either, and a member is never equal to its plain value. That last point is what makes Enums safe, and it's where today starts.

## A fixed set of choices

Without Enums, a program usually represents choices with strings or numbers:

Every call ran without complaint, and three of them gave a quietly wrong answer. Nothing says which strings are allowed, and a typo is just another string. With an Enum, the allowed choices are written down once, and a typo fails loudly:

What a class that inherits from `Enum` gives you:

- Each line `NAME = value` makes one **member**. Members are written in CAPITALS, like constants.
- Each member exists exactly once, so compare members with `is` (or `==`, which does the same here).
- The class can be iterated (`for light in Light`, in the order written), has a `len`, and supports `in`.
- Members can't be reassigned, and you can't make new ones: `Light()` and `Light.PURPLE = 4` both fail.
- Members are hashable, so they make good dictionary keys and set items.
- A member isn't equal to its value: `Light.GREEN == "green"` is False. Code must say `Light.GREEN`, which is the point.

## Looking members up

Data from outside (a file, a form, JSON) arrives as plain values or names. Turn it into a member at the edge of your program, then use members everywhere inside:

- `Size(value)` finds a member by **value**, raising `ValueError` if there isn't one.
- `Size[name]` finds a member by **name**, raising `KeyError`, like a dictionary.
- Both are exact: `"large"` isn't `"L"` or `"LARGE"`. Tidy the text first (`.strip().upper()`) if your input needs it.

If two members have the **same value**, the second is an **alias** for the first, not a separate member. That's occasionally handy (`MONDAY = 1` and `MON = 1`) but usually a mistake. Put `@unique` (from `enum`) above the class to make Python refuse duplicate values.

## Enums with behaviour

An Enum is a class, so it can have methods and properties. Inside a method, `self` is a member:

Tables that belong to the members can live in a dictionary keyed by member, which reads well and keeps the Enum itself simple:

Plain Enum members have **no order**: `Size.SMALL < Size.LARGE` raises `TypeError`, because Python can't know whether the values (here strings) mean anything ordered. If the order of the members matters, use their position in `list(Size)`, or a key such as `sorted(sizes, key=list(Size).index)`.

The `auto()` helper fills in values for you, when the values don't matter:

## Enums in data

Enums fit neatly into dataclasses: the field's annotation names the Enum, and the default is a member. When saving to JSON (which only knows strings, numbers, lists and dictionaries), store `member.value` or `member.name`, and turn it back into a member when loading:

Inside the program, `o.stage is Stage.PAID` can't be misspelt without an error. At the edges, `.value` goes out and `Stage(...)` comes back in.

## IntEnum, StrEnum and Flag

Three special kinds of Enum give up some of the safety for convenience:

| Kind | Members are also... | Use it when |
|---|---|---|
| `IntEnum` | real `int`s: equal to numbers, ordered, usable in arithmetic | code or files already use the numbers, and members need an order |
| `StrEnum` | real `str`s: equal to their values, usable wherever text is | values go straight into text, JSON or files |
| `Flag` | combinable with `|` into sets of options | several options can be on at once, like permissions |

The trade-off with `IntEnum` and `StrEnum` is that a typo'd plain value can sneak back in: `Priority.NORMAL == 2` is True, so a stray `2` elsewhere is treated as the member. Use plain `Enum` by default, and the others when you have a reason.

`Flag` members combine like a set: `Perm.READ | Perm.WRITE` holds both, `in` checks whether an option is included, and `&` keeps the options two flags share. `auto()` gives flags the values 1, 2, 4, 8, so they combine without overlapping.

## Practice

## Quiz

## Daily challenge

## Before you go

Search your own code from earlier weeks for strings or numbers that stand for a fixed set of choices (a status, a category, a mode) and decide which would be safer as an Enum. Tomorrow: `NamedTuple`, the typed, class-based cousin of Week 20's `namedtuple`.